In [1]:
import numpy as np
from pathlib import Path 
import matplotlib.pyplot as plt
import os
import pandas as pd

In [2]:
true_mi = {20: 5,
           40: 10, 
           80: 20, 
           160: 40, 
           320: 80}

## Get metrics for scRatio

In [3]:
result_dir = Path("../../project_folder/results/mi_experiments/scRatio/simulate_under_uncond_sweep/")

results = {"Config": [], 
           "MSE_test": [],
           "MSE_val": [], 
           "Dimensions": [], 
           "Run": []
          }


for file in os.listdir(result_dir):
    for dim in [20, 40, 80, 160, 320]:
        # Get val and test ratios 
        ratios_val = np.load(result_dir / file / f"ratios_{dim}_val.npy")
        ratios_test = np.load(result_dir / file / f"ratios_{dim}_test.npy")
        # Split into three
        ratios_split_val = np.split(ratios_val, 3)
        ratios_split_test = np.split(ratios_test, 3)
        
        split_avgs_val = [arr.mean() for arr in ratios_split_val]
        split_avgs_test = [arr.mean() for arr in ratios_split_test]
        
        # MSE with real values 
        split_avgs_val_mse = [np.abs(avg - true_mi[dim]) for avg in split_avgs_val]
        split_avgs_test_mse = [np.abs(avg - true_mi[dim]) for avg in split_avgs_test]
        for i in range(len(split_avgs_val_mse)):
            results["Config"].append(file)
            results["MSE_val"].append(split_avgs_val_mse[i])
            results["MSE_test"].append(split_avgs_test_mse[i])
            results["Dimensions"].append(dim)
            results["Run"].append(i)

In [4]:
results_df = pd.DataFrame(results)
results_df.groupby(["Config", "Dimensions"]).mean()

MSE_test    MSE_val  Run
Config          Dimensions                           
deterministic   20           0.048347   0.047935  1.0
                40           0.053398   0.043350  1.0
                80           0.051510   0.050333  1.0
                160          0.164042   0.150202  1.0
                320         21.182093  21.090273  1.0
sigmamin_0.01   20           0.043664   0.068197  1.0
                40           0.061230   0.051906  1.0
                80           0.069777   0.060178  1.0
                160          0.133917   0.026206  1.0
                320         20.854738  20.781925  1.0
sigmamin_0.1    20           0.043552   0.057800  1.0
                40           0.030456   0.028866  1.0
                80           0.024389   0.012668  1.0
                160          0.164224   0.082579  1.0
                320         21.000315  20.946859  1.0
stochastic_0.1  20           0.028488   0.033410  1.0
                40           0.051957   0.055336  1.0
                80           0.026578   0.029005  1.0
                160          0.259168   0.234187  1.0
                320         14.579456  14.442523  1.0
stochastic_0.25 20           0.027236   0.040407  1.0
                40           0.093585   0.093981  1.0
                80           0.071206   0.079332  1.0
                160          0.105029   0.146282  1.0
                320          1.157954   1.241575  1.0
stochastic_0.5  20           0.019956   0.027286  1.0
                40           0.217669   0.254589  1.0
                80           0.311665   0.322271  1.0
                160          0.179095   0.138977  1.0
                320          1.096039   0.949196  1.0

In [5]:
results_df.groupby(["Config", "Dimensions"]).std() / np.sqrt(3)

MSE_test   MSE_val      Run
Config          Dimensions                             
deterministic   20          0.018924  0.031030  0.57735
                40          0.020265  0.008825  0.57735
                80          0.025177  0.025236  0.57735
                160         0.045341  0.046130  0.57735
                320         0.137631  0.128294  0.57735
sigmamin_0.01   20          0.013453  0.014492  0.57735
                40          0.033590  0.022014  0.57735
                80          0.034594  0.025879  0.57735
                160         0.019973  0.010259  0.57735
                320         0.045957  0.087593  0.57735
sigmamin_0.1    20          0.020456  0.025572  0.57735
                40          0.018748  0.014548  0.57735
                80          0.009211  0.002311  0.57735
                160         0.066919  0.061354  0.57735
                320         0.260600  0.266705  0.57735
stochastic_0.1  20          0.012318  0.021736  0.57735
                40          0.020143  0.021511  0.57735
                80          0.017515  0.000559  0.57735
                160         0.098208  0.141585  0.57735
                320         2.090012  2.123522  0.57735
stochastic_0.25 20          0.012435  0.013443  0.57735
                40          0.026821  0.045427  0.57735
                80          0.043744  0.049857  0.57735
                160         0.031021  0.062660  0.57735
                320         0.268263  0.253932  0.57735
stochastic_0.5  20          0.007249  0.007990  0.57735
                40          0.092419  0.082423  0.57735
                80          0.024924  0.032523  0.57735
                160         0.055980  0.111642  0.57735
                320         0.348399  0.326402  0.57735

## Results baseline DRECFM

Results for the competing models were obtained using the code at https://github.com/ksnxr/dre-prob-paths using the "TwoSB" option. The adapted repo will be made available upon request, message alessandro.palma@helmholtz-munich.de.

In [6]:
path_dre_cfm = Path("../../project_folder/results/mi_experiments/dre-cfm/dre-cfm_sb_retrains/")

In [7]:
results_baseline_cfm  = {"MSE": [],
                        "Dimensions": [], 
                        "Run": []
                       }

In [8]:
for file in os.listdir(path_dre_cfm):
    file_split = file.split("_")  
    dimensions, rep = int(file_split[4]), int(file_split[6])
    results_baseline_cfm["Run"].append(rep)
    results_baseline_cfm["Dimensions"].append(dimensions)
    
    ratios_cfm = np.load(path_dre_cfm / file / "metrics" / "estim_logr.npy")
    ratio_mean = ratios_cfm.mean()
    results_baseline_cfm["MSE"].append(np.abs(ratio_mean - true_mi[dimensions]))

In [9]:
results_baseline_cfm_df = pd.DataFrame(results_baseline_cfm)

In [10]:
results_baseline_cfm_df.groupby("Dimensions").mean()

,MSE,Run
Dimensions,,
20,0.061192,40.333333
40,0.085309,40.333333
80,0.230340,40.333333
160,0.872691,40.333333
320,2.154556,40.333333


In [11]:
results_baseline_cfm_df.groupby("Dimensions").std() / np.sqrt(3)

,MSE,Run
Dimensions,,
20,0.008241,22.805945
40,0.043984,22.805945
80,0.034371,22.805945
160,0.101788,22.805945
320,0.078399,22.805945


## Results baseline TSM

In [12]:
path_dre_tsm = Path("../../project_folder/results/mi_experiments/dre-cfm/dre-cfm_tsm_retrains/")

In [13]:
results_baseline_tsm = {"MSE": [],
                        "Dimensions": [], 
                        "Run": []
                       }

In [14]:
for file in os.listdir(path_dre_tsm):
    file_split = file.split("_")  
    dimensions, rep = int(file_split[4]), int(file_split[6])
    results_baseline_tsm["Run"].append(rep)
    results_baseline_tsm["Dimensions"].append(dimensions)
    
    ratios_cfm = np.load(path_dre_tsm / file / "metrics" / "estim_logr.npy")
    ratio_mean = ratios_cfm.mean()
    results_baseline_tsm["MSE"].append(np.abs(ratio_mean - true_mi[dimensions]))

In [15]:
results_baseline_tsm_df = pd.DataFrame(results_baseline_tsm)

In [16]:
results_baseline_tsm_df.groupby("Dimensions").mean()

,MSE,Run
Dimensions,,
20,0.034667,40.333333
40,0.382803,40.333333
80,0.250295,40.333333
160,0.888362,40.333333
320,3.546334,40.333333


In [17]:
results_baseline_tsm_df.groupby("Dimensions").std() / np.sqrt(3)

,MSE,Run
Dimensions,,
20,0.010257,22.805945
40,0.023554,22.805945
80,0.101173,22.805945
160,0.108121,22.805945
320,0.120376,22.805945


## Results baseline classifier 

In [18]:
path_clf_results = Path("../../project_folder/results/mi_experiments/classifier/mi")

In [19]:
results_baseline_clf = {"MSE_test": [],
                        "MSE_val": [],
                        "Dimensions": [], 
                        "Hidden dim": [],
                        "Hidden layers": [], 
                        "Iteration": []
                       }

In [20]:
for file in os.listdir(path_clf_results):
    file_split = file.split("_")
    input_dim, hidden_dim, hidden_layers, iteration = int(file_split[1]), int(file_split[4]), int(file_split[8]), int(file_split[10]) 
    
    results_path = path_clf_results / file
    log_ratios_val = np.load(results_path / "log_ratios_val.npy")
    log_ratios_test = np.load(results_path / "log_ratios_test.npy")

    log_ratios_val_mean = log_ratios_val.mean()
    log_ratios_test_mean = log_ratios_test.mean()
    
    mse_val = np.abs(log_ratios_val_mean - true_mi[input_dim])
    mse_test = np.abs(log_ratios_test_mean - true_mi[input_dim])

    results_baseline_clf["MSE_test"].append(mse_test)
    results_baseline_clf["MSE_val"].append(mse_val)
    results_baseline_clf["Dimensions"].append(input_dim)
    results_baseline_clf["Hidden dim"].append(hidden_dim)
    results_baseline_clf["Hidden layers"].append(hidden_layers)
    results_baseline_clf["Iteration"].append(iteration)

In [21]:
results_baseline_clf_df = pd.DataFrame(results_baseline_clf)

In [22]:
results_baseline_clf_df.groupby(["Hidden dim", "Hidden layers", "Dimensions"]).mean()

MSE_test    MSE_val  Iteration
Hidden dim Hidden layers Dimensions                                 
64         0             20           4.999091   4.998561        1.0
                         40          10.001134  10.001160        1.0
                         80          20.002754  20.003859        1.0
                         160         40.000824  40.000675        1.0
                         320         80.000916  80.000023        1.0
           1             20           0.269195   0.250155        1.0
                         40          10.178844  10.208176        1.0
                         80          20.249241  20.332100        1.0
                         160          2.618514   2.584152        1.0
                         320         45.561848  45.465717        1.0
           2             20           0.650033   0.628907        1.0
                         40          27.802460  28.004198        1.0
                         80          27.963964  28.078140        1.0
                         160          6.016335   5.946677        1.0
                         320         39.652370  39.490101        1.0
           3             20           2.010048   2.014922        1.0
                         40          29.840891  30.011673        1.0
                         80          29.689951  29.691900        1.0
                         160          5.002014   4.820521        1.0
                         320         36.203735  36.213131        1.0
128        0             20           4.999945   4.999830        1.0
                         40          10.002879  10.002650        1.0
                         80          19.999985  20.001337        1.0
                         160         39.997551  39.997074        1.0
                         320         80.006523  80.004189        1.0
           1             20           0.292107   0.283365        1.0
                         40          17.874338  17.937006        1.0
                         80          20.831427  20.903448        1.0
                         160          8.700425   8.657140        1.0
                         320         51.271667  51.256649        1.0
           2             20           2.202716   2.183934        1.0
                         40          31.451254  31.514704        1.0
                         80          28.578684  28.633551        1.0
                         160          1.462114   1.469842        1.0
                         320         45.021198  44.936695        1.0
           3             20          22.333433  22.204453        1.0
                         40          66.868324  67.095520        1.0
                         80          26.612513  26.684080        1.0
                         160          0.829197   0.827524        1.0
                         320         30.721001  30.759466        1.0
256        0             20           4.994571   4.994337        1.0
                         40           9.997557   9.997312        1.0
                         80          19.996721  19.997435        1.0
                         160         40.002094  40.002640        1.0
                         320         79.996155  79.996338        1.0
           1             20           0.309518   0.296025        1.0
                         40          19.846458  19.921377        1.0
                         80          19.512045  19.547825        1.0
                         160         10.403821  10.405325        1.0
                         320         53.082352  53.066120        1.0
           2             20          10.586574  10.570610        1.0
                         40          52.805080  53.022686        1.0
                         80          27.516792  27.538584        1.0
                         160          3.836000   3.886150        1.0
                         320         46.868725  46.789341        1.0
           3             20          31.327326  31.180923        1.0
                         40         

In [ ]:
df_mean = (
    results_baseline_clf_df
    .groupby(["Hidden dim", "Hidden layers", "Dimensions"], as_index=False)
    .mean()   # averages across iteration and any other non-grouped numeric columns
)

df_std = (
    results_baseline_clf_df
    .groupby(["Hidden dim", "Hidden layers", "Dimensions"], as_index=False)
    .std()   # averages across iteration and any other non-grouped numeric columns
) / np.sqrt(3)

best_configs = df_mean.loc[
    df_mean.groupby("Dimensions")["MSE_val"].idxmin()
]

best_configs_std = df_std.loc[
    df_mean.groupby("Dimensions")["MSE_val"].idxmin()
]

In [24]:
best_configs

,Hidden dim,Hidden layers,Dimensions,MSE_test,MSE_val,Iteration
5,64,1,20,0.269195,0.250155,1.0
41,256,0,40,9.997557,9.997312,1.0
47,256,1,80,19.512045,19.547825,1.0
38,128,3,160,0.829197,0.827524,1.0
59,256,3,320,26.681280,26.527079,1.0


In [25]:
best_configs_std

,Hidden dim,Hidden layers,Dimensions,MSE_test,MSE_val,Iteration
5,36.950417,0.577350,11.547005,0.036251,0.032080,0.57735
41,147.801669,0.000000,23.094011,0.002703,0.002613,0.57735
47,147.801669,0.577350,46.188022,0.180991,0.163381,0.57735
38,73.900834,1.732051,92.376043,0.109046,0.088112,0.57735
59,147.801669,1.732051,184.752086,3.696742,3.715490,0.57735
